In [1]:
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import label
from skimage.filters import threshold_otsu, gaussian
from skimage.morphology import remove_small_objects, closing, disk, erosion
from skimage.draw import polygon2mask
from skimage.measure import regionprops_table
from skimage.segmentation import watershed
from skimage.morphology import local_maxima

from scipy import ndimage as ndi

import os

In [2]:
mask = tifffile.imread(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\C2_02_msg_103-1_col1_empty_cd8.tif')

In [3]:
blurred = gaussian(mask, sigma=1.5)
thresh = threshold_otsu(blurred)
binary = blurred > thresh*0.8

In [4]:
tifffile.imwrite(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\C2_02_msg_103-1_col1_empty_cd8_binary.tif', binary)

In [6]:
tissue_polygon = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\tissue_boundary.csv', index_col=0)

In [7]:
mask_shape = mask.shape
tissue_mask = polygon2mask(mask_shape, tissue_polygon.values.astype('int'))

In [8]:
# tifffile.imwrite(r'Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_126-1\00_region_segmentation\tissue_mask.tif', mask)

In [10]:
compartment_mask = tissue_mask & ~binary
# compartment_mask = watershed(compartment_mask)

In [11]:
tifffile.imwrite(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\compartment_mask.tif', compartment_mask)

In [12]:
# distance = ndi.distance_transform_edt(compartment_mask)
# markers, _ = ndi.label(local_maxima(distance))
# compartments = watershed(-distance, markers, mask=compartment_mask)

In [13]:
# erode then relable
compartment_mask = erosion(compartment_mask, disk(20))

In [14]:
tifffile.imwrite(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\watershed.tif', compartment_mask)

In [15]:
compartments = label(compartment_mask)
compartments_clean = remove_small_objects(compartments, min_size=3000)

In [16]:
# props = regionprops_table(compartments_clean, properties=['label', 'area', 'eccentricity'])
# props_df = pd.DataFrame(props)

# bad_labels = props_df.query("area < 1000 or eccentricity > 0.98")["label"].values
# for lbl in bad_labels:
#     compartments_clean[compartments_clean == lbl] = 0

In [17]:
np.unique(compartments_clean).shape

(763,)

In [18]:
tifffile.imwrite(r'Y:\coskun-lab2\Zhou\12_MSG\20241206_ssa-_gc\103-1\00_region_segmentation\filtered_compartments.tif', compartments_clean)